# Visualizing Education Metrics with Seaborn

**A practical guide to the most useful Seaborn plots for education analytics**  
Focus: Student achievement, equity gaps, school performance, and learning outcomes.

This notebook complements the previous *NumPy + Pandas Education* project.  
Here we concentrate on **clear, publication-ready visualizations** that help educators and analysts communicate insights quickly.

### What you will learn to visualize
| Metric / Question | Recommended Seaborn Approach |
|-------------------|------------------------------|
| Score distributions by subject | `histplot`, `kdeplot`, `violinplot` |
| Achievement gaps (FRPL, ELL, gender…) | `boxplot`, `violinplot`, `barplot`, `pointplot` |
| School comparisons | `barplot`, `stripplot`, `swarmplot` |
| Subject correlations | `heatmap`, `pairplot`, `scatterplot` |
| Multi-dimensional views | `catplot`, `FacetGrid`, `relplot` |
| Longitudinal trends | `lineplot` |
| Rankings & matrices | `heatmap`, `clustermap` |

> All plots are built on realistic synthetic student-level data so you can adapt the code directly to real district or state assessment files.


## 1. Setup & Data Generation


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Clean, readable defaults for education reports
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.05)
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['axes.labelsize'] = 11

rng = np.random.default_rng(42)

# ------------------------------------------------------------------
# Generate realistic student-level data (1,200 students, 6 schools)
# ------------------------------------------------------------------
n = 1200
schools = [f"School {i}" for i in range(1, 7)]
school_id = rng.choice(schools, n, p=[0.22, 0.18, 0.17, 0.16, 0.15, 0.12])

# Base ability + school effect + noise
school_effect = {s: v for s, v in zip(schools, [4.5, 1.2, -0.8, -2.5, 2.0, -3.8])}
base = 70 + np.array([school_effect[s] for s in school_id]) + rng.normal(0, 9, n)

math    = np.clip(base + rng.normal(0, 6, n) - 1.5, 25, 100)
reading = np.clip(base + rng.normal(0, 5.5, n) + 1.0, 28, 100)
science = np.clip(base + rng.normal(0, 7, n) - 3.0, 20, 100)

# Demographics
gender = rng.choice(["Female", "Male"], n)
ell    = rng.choice([False, True], n, p=[0.81, 0.19])
sped   = rng.choice([False, True], n, p=[0.87, 0.13])
frpl   = rng.choice([False, True], n, p=[0.52, 0.48])   # Free/Reduced Price Lunch

# Mild demographic effects (for realistic gaps)
math    = np.clip(math    - 4.5*frpl - 3.0*ell - 5.5*sped + rng.normal(0, 1, n), 20, 100)
reading = np.clip(reading - 3.8*frpl - 5.5*ell - 4.0*sped + rng.normal(0, 1, n), 20, 100)
science = np.clip(science - 4.2*frpl - 2.5*ell - 6.0*sped + rng.normal(0, 1, n), 20, 100)

students = pd.DataFrame({
    "student_id": np.arange(10001, 10001 + n),
    "school": school_id,
    "gender": gender,
    "ell": ell,
    "sped": sped,
    "frpl": frpl,
    "math": math.round(1),
    "reading": reading.round(1),
    "science": science.round(1),
})
students["avg_score"] = students[["math", "reading", "science"]].mean(axis=1).round(1)
students["at_risk"] = students["avg_score"] < 55

# Long-form version (very useful for many Seaborn plots)
scores_long = students.melt(
    id_vars=["student_id", "school", "gender", "ell", "sped", "frpl", "at_risk"],
    value_vars=["math", "reading", "science"],
    var_name="subject",
    value_name="score"
)

print("Student-level data ready")
print(f"Shape: {students.shape}")
print(f"At-risk rate: {students['at_risk'].mean()*100:.1f}%")
display(students.head())


## 2. Understanding Score Distributions

Before comparing groups we need a clear picture of the overall shape of achievement.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Histogram + KDE overlay
sns.histplot(data=scores_long, x="score", hue="subject", kde=True,
             element="step", stat="density", common_norm=False, ax=axes[0])
axes[0].set_title("Score Distributions by Subject")
axes[0].set_xlabel("Score")

# Violin + box combination (shows density + quartiles)
sns.violinplot(data=scores_long, x="subject", y="score", inner="quartile",
               palette="Set2", ax=axes[1])
axes[1].set_title("Score Distribution Shape by Subject")
axes[1].set_xlabel("")
axes[1].set_ylabel("Score")

plt.tight_layout()
plt.show()

print("Insight: Science has a wider spread and a lower center than Reading.")
print("This often signals greater variability in science instruction or curriculum alignment.")


## 3. Visualizing Equity Gaps

One of the most important uses of visualization in education is to make achievement gaps visible and actionable.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Box plot – classic gap view
sns.boxplot(data=students, x="frpl", y="avg_score", hue="frpl",
            palette={False: "#4c72b0", True: "#c44e52"}, ax=axes[0], legend=False)
axes[0].set_xticklabels(["Non-FRPL", "FRPL"])
axes[0].set_title("Overall Average Score by FRPL Status")
axes[0].set_xlabel("Free / Reduced Price Lunch")
axes[0].set_ylabel("Average Score")

# Violin for richer distributional comparison
sns.violinplot(data=scores_long, x="subject", y="score", hue="frpl",
               split=True, inner="quart", palette={False: "#4c72b0", True: "#c44e52"},
               ax=axes[1])
axes[1].set_title("Subject Scores by FRPL (split violin)")
axes[1].set_xlabel("")
axes[1].legend(title="FRPL", labels=["No", "Yes"])

plt.tight_layout()
plt.show()


In [ ]:
# Point plot – excellent for showing means + confidence intervals across multiple groups
fig, ax = plt.subplots(figsize=(10, 5))
sns.pointplot(data=scores_long, x="subject", y="score", hue="frpl",
              errorbar=("ci", 95), dodge=True, markers=["o", "s"],
              palette={False: "#4c72b0", True: "#c44e52"}, ax=ax)
ax.set_title("Mean Score ± 95% CI by Subject and FRPL Status")
ax.set_xlabel("")
ax.set_ylabel("Mean Score")
ax.legend(title="FRPL")
plt.tight_layout()
plt.show()

print("Insight: The FRPL gap is present in every subject and is largest in Math and Science.")
print("Point plots make the size of the gap and the uncertainty around it immediately visible.")


## 4. Comparing Schools


In [ ]:
# Ordered bar plot of school means
school_order = students.groupby("school")["avg_score"].mean().sort_values(ascending=False).index

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

sns.barplot(data=students, x="school", y="avg_score", order=school_order,
            palette="Blues_r", errorbar=("ci", 95), ax=axes[0])
axes[0].set_title("Mean Average Score by School (95% CI)")
axes[0].set_xlabel("")
axes[0].set_ylabel("Mean Score")
axes[0].tick_params(axis="x", rotation=30)

# Strip plot – shows every student (good for spotting outliers and spread)
sns.stripplot(data=students, x="school", y="avg_score", order=school_order,
              size=3, alpha=0.45, color="steelblue", ax=axes[1])
sns.boxplot(data=students, x="school", y="avg_score", order=school_order,
            showfliers=False, width=0.35, boxprops=dict(alpha=0.3), ax=axes[1])
axes[1].set_title("Student-level Scores by School")
axes[1].set_xlabel("")
axes[1].tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.show()


## 5. Relationships Between Subjects


In [ ]:
# Correlation heatmap
corr = students[["math", "reading", "science", "avg_score"]].corr()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu_r", center=0,
            square=True, linewidths=0.5, ax=axes[0])
axes[0].set_title("Subject Correlation Matrix")

# Scatter with regression line
sns.regplot(data=students, x="math", y="reading", scatter_kws={"alpha": 0.35, "s": 18},
            line_kws={"color": "darkred"}, ax=axes[1])
axes[1].set_title("Math vs Reading (with linear fit)")
axes[1].set_xlabel("Math Score")
axes[1].set_ylabel("Reading Score")

plt.tight_layout()
plt.show()

print("Insight: Math and Reading are strongly related (r ≈ 0.75–0.85 in most systems).")
print("Students weak in one are frequently weak in the other → integrated intervention design.")


In [ ]:
# Pair plot – quick multivariate overview (sample for speed)
sample = students.sample(400, random_state=42)
sns.pairplot(sample, vars=["math", "reading", "science"],
             hue="frpl", corner=True, plot_kws={"alpha": 0.5, "s": 22},
             palette={False: "#4c72b0", True: "#c44e52"})
plt.suptitle("Pairwise Relationships by FRPL Status", y=1.02)
plt.show()


## 6. Faceted and Multi-Dimensional Views

When we need to examine gaps across several factors at once, Seaborn’s categorical plotting functions shine.


In [ ]:
# catplot – powerful high-level interface
g = sns.catplot(
    data=scores_long, kind="box",
    x="subject", y="score", hue="frpl",
    col="ell",
    palette={False: "#4c72b0", True: "#c44e52"},
    height=4.5, aspect=0.9
)
g.set_axis_labels("", "Score")
g.set_titles("ELL: {col_name}")
g.legend.set_title("FRPL")
plt.suptitle("Score Distributions by Subject, FRPL and ELL Status", y=1.03)
plt.show()


In [ ]:
# Another useful view: at-risk rate by school and FRPL
at_risk_rate = (students
                .groupby(["school", "frpl"])["at_risk"]
                .mean()
                .reset_index())

fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(data=at_risk_rate, x="school", y="at_risk", hue="frpl",
            order=school_order,
            palette={False: "#4c72b0", True: "#c44e52"}, ax=ax)
ax.set_title("At-Risk Rate by School and FRPL Status")
ax.set_ylabel("Proportion At-Risk")
ax.set_xlabel("")
ax.legend(title="FRPL", labels=["No", "Yes"])
ax.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.show()


## 7. Heatmaps for School × Subject Performance


In [ ]:
# School × Subject mean score matrix
school_subj = scores_long.pivot_table(
    values="score", index="school", columns="subject", aggfunc="mean"
).reindex(school_order)

fig, ax = plt.subplots(figsize=(8, 5))
sns.heatmap(school_subj, annot=True, fmt=".1f", cmap="YlGnBu",
            linewidths=0.5, ax=ax)
ax.set_title("Mean Score by School and Subject")
ax.set_xlabel("")
ax.set_ylabel("")
plt.tight_layout()
plt.show()

# Optional: clustermap to reveal natural groupings of schools
sns.clustermap(school_subj, annot=True, fmt=".1f", cmap="YlGnBu",
               figsize=(7, 6), linewidths=0.4)
plt.suptitle("Clustered School × Subject Performance", y=1.02)
plt.show()


### Additional Practical Heatmaps for Education Metrics

Heatmaps are one of the most effective ways to show multi-dimensional education data at a glance. Below are three high-value variants you can reuse in real district reports.


In [ ]:
# ------------------------------------------------------------------
# 1. Proficiency Rate Heatmap (School × Subject)
# ------------------------------------------------------------------
# Define proficiency as score >= 70
prof = scores_long.copy()
prof["proficient"] = prof["score"] >= 70

prof_rate = (prof
             .groupby(["school", "subject"])["proficient"]
             .mean()
             .unstack()
             .reindex(school_order) * 100)

fig, ax = plt.subplots(figsize=(8, 5))
sns.heatmap(
    prof_rate,
    annot=True, fmt=".0f",
    cmap="YlGnBu",
    linewidths=0.6,
    linecolor="white",
    cbar_kws={"label": "% Proficient"},
    ax=ax
)
ax.set_title("Proficiency Rate (%) by School and Subject\n(cut-score = 70)", pad=12)
ax.set_xlabel("")
ax.set_ylabel("")
plt.tight_layout()
plt.show()

print("Insight: Darker cells = higher proficiency. Quickly reveals which school-subject")
print("combinations need the most instructional attention.")


In [ ]:
# ------------------------------------------------------------------
# 2. Achievement Gap Heatmap (FRPL gap by School × Subject)
# ------------------------------------------------------------------
# Gap = Non-FRPL mean − FRPL mean
gap_data = (scores_long
            .groupby(["school", "subject", "frpl"])["score"]
            .mean()
            .unstack("frpl"))

col_non = False if False in gap_data.columns else "False"
col_frpl = True if True in gap_data.columns else "True"
gap_data["gap"] = gap_data[col_non] - gap_data[col_frpl]   # Non-FRPL − FRPL
gap_matrix = gap_data["gap"].unstack("subject").reindex(school_order)

fig, ax = plt.subplots(figsize=(8, 5))
sns.heatmap(
    gap_matrix,
    annot=True, fmt=".1f",
    cmap="RdYlGn_r",          # red = larger gap (worse)
    center=0,
    linewidths=0.6,
    linecolor="white",
    cbar_kws={"label": "Score Gap (points)"},
    ax=ax
)
ax.set_title("FRPL Achievement Gap by School and Subject\n(Non-FRPL mean − FRPL mean)", pad=12)
ax.set_xlabel("")
ax.set_ylabel("")
plt.tight_layout()
plt.show()

print("Insight: Positive values = Non-FRPL students score higher.")
print("This view is especially powerful for equity discussions with school leaders.")


In [ ]:
# ------------------------------------------------------------------
# 3. Annotated Correlation Heatmap with better styling
# ------------------------------------------------------------------
corr = students[["math", "reading", "science", "avg_score"]].corr()

# Create a mask for the upper triangle (cleaner look)
mask = np.triu(np.ones_like(corr, dtype=bool))

fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(
    corr,
    mask=mask,
    annot=True, fmt=".2f",
    cmap="RdBu_r",
    center=0,
    square=True,
    linewidths=1.2,
    linecolor="white",
    cbar_kws={"shrink": 0.8, "label": "Pearson r"},
    annot_kws={"size": 12},
    ax=ax
)
ax.set_title("Subject Correlation Matrix (lower triangle)", pad=12)
plt.tight_layout()
plt.show()


In [ ]:
# ------------------------------------------------------------------
# 4. At-Risk Rate Heatmap (School × Demographic Group)
# ------------------------------------------------------------------
# Create a combined demographic label for illustration
students["group"] = np.select(
    [
        students["frpl"] & students["ell"],
        students["frpl"] & ~students["ell"],
        ~students["frpl"] & students["ell"],
        ~students["frpl"] & ~students["ell"]
    ],
    ["FRPL + ELL", "FRPL only", "ELL only", "Neither"],
    default="Other"
)

risk_by_group = (students
                 .groupby(["school", "group"])["at_risk"]
                 .mean()
                 .unstack()
                 .reindex(school_order) * 100)

# Reorder columns for logical reading
col_order = ["Neither", "ELL only", "FRPL only", "FRPL + ELL"]
risk_by_group = risk_by_group[[c for c in col_order if c in risk_by_group.columns]]

fig, ax = plt.subplots(figsize=(9, 5))
sns.heatmap(
    risk_by_group,
    annot=True, fmt=".0f",
    cmap="OrRd",
    linewidths=0.6,
    linecolor="white",
    cbar_kws={"label": "% At-Risk"},
    ax=ax
)
ax.set_title("At-Risk Rate (%) by School and Student Group", pad=12)
ax.set_xlabel("")
ax.set_ylabel("")
plt.tight_layout()
plt.show()

print("Insight: Intersectionality matters. Students who are both FRPL and ELL")
print("often show the highest at-risk rates — important for targeted support design.")


## 8. Trends Over Time


In [ ]:
# Simulate 5-year district trend
years = [2019, 2020, 2021, 2022, 2023]
trend = pd.DataFrame({
    "year": years * 3,
    "subject": np.repeat(["math", "reading", "science"], 5),
    "mean_score": [
        68.2, 67.5, 69.1, 71.0, 72.4,   # math
        71.5, 70.8, 72.0, 73.6, 74.3,   # reading
        65.8, 64.9, 66.5, 68.2, 69.7    # science
    ]
})

fig, ax = plt.subplots(figsize=(10, 5))
sns.lineplot(data=trend, x="year", y="mean_score", hue="subject",
             marker="o", linewidth=2.2, ax=ax)
ax.set_title("District Mean Scores Over Time")
ax.set_ylabel("Mean Score")
ax.set_xlabel("Year")
ax.legend(title="Subject")
plt.tight_layout()
plt.show()

print("Insight: All subjects show recovery after 2020, with Math and Science still lagging Reading.")


## 9. Putting It Together – A Compact Equity & Performance Dashboard


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 10))

# 1. Overall distributions
sns.kdeplot(data=scores_long, x="score", hue="subject", fill=True,
            common_norm=False, alpha=0.4, ax=axes[0, 0])
axes[0, 0].set_title("Score Density by Subject")
axes[0, 0].set_xlabel("Score")

# 2. FRPL gap by subject
sns.pointplot(data=scores_long, x="subject", y="score", hue="frpl",
              errorbar=("ci", 95), dodge=True, ax=axes[0, 1],
              palette={False: "#4c72b0", True: "#c44e52"})
axes[0, 1].set_title("FRPL Achievement Gap by Subject")
axes[0, 1].set_xlabel("")
axes[0, 1].legend(title="FRPL")

# 3. School ranking
sns.barplot(data=students, x="avg_score", y="school", order=school_order,
            palette="Blues_r", errorbar=("ci", 95), ax=axes[1, 0])
axes[1, 0].set_title("Schools Ranked by Mean Average Score")
axes[1, 0].set_xlabel("Mean Score")
axes[1, 0].set_ylabel("")

# 4. At-risk heatmap proxy
risk_pivot = students.pivot_table(values="at_risk", index="school",
                                  columns="frpl", aggfunc="mean")
risk_pivot.columns = ["Non-FRPL", "FRPL"]
sns.heatmap(risk_pivot.reindex(school_order), annot=True, fmt=".1%",
            cmap="OrRd", ax=axes[1, 1])
axes[1, 1].set_title("At-Risk Rate by School × FRPL")
axes[1, 1].set_xlabel("")

plt.tight_layout()
plt.show()


## 10. Key Insights from the Visualizations

| Finding | Visual Evidence | Implication |
|---------|-----------------|-------------|
| Science is the weakest / most variable subject | KDE + violin plots | Prioritize science instructional support |
| Consistent FRPL gaps across all subjects | Point plots + split violins | Gaps are systemic, not subject-specific |
| Large school-to-school variation | Ranked bar + strip plots | Need both school improvement and equity strategies |
| Strong Math–Reading correlation | Heatmap + scatter | Early literacy & numeracy interventions reinforce each other |
| At-risk concentration in certain schools + FRPL | Heatmap of at-risk rates | Targeted resource allocation opportunity |

### Seaborn Best Practices for Education Reports
1. **Prefer `pointplot` or `barplot` with error bars** when showing group means — they communicate uncertainty.
2. **Use split violins or box + strip** when the distribution shape (not just the mean) matters.
3. **Order categories meaningfully** (e.g., schools by performance) instead of alphabetical order.
4. **Facet by the second or third variable** (`col=`, `row=`) rather than overloading a single plot with too many hues.
5. **Keep color consistent** across a report (e.g., always the same color for FRPL = Yes).
6. **Annotate heatmaps** when the audience needs exact values; otherwise let color do the work.

---

### Next Steps
- Replace the synthetic data with your real assessment file (same column structure works).
- Add student growth percentiles or prior-year scores for value-added style views.
- Export any figure with `plt.savefig(..., dpi=300, bbox_inches="tight")` for presentations or board reports.

**You now have a ready-to-adapt toolkit of Seaborn visualizations specifically tuned for education metrics.**


---
# 11. Full Python Code Blocks (Copy-Paste Ready)

The cells below contain **complete, self-contained code**.  
Each block includes the necessary imports, data preparation (or assumes the `students` / `scores_long` DataFrames already exist), and a polished visualization.

You can copy any block into a new notebook or script and run it with minimal changes.


### Full Block 1 — Data Generation + Score Distributions


In [ ]:
# ================================================================
# FULL CODE: Generate education data + plot score distributions
# ================================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="muted", font_scale=1.05)
rng = np.random.default_rng(42)

# ----- Data generation -----
n = 1200
schools = [f"School {i}" for i in range(1, 7)]
school_id = rng.choice(schools, n, p=[0.22, 0.18, 0.17, 0.16, 0.15, 0.12])
school_effect = {s: v for s, v in zip(schools, [4.5, 1.2, -0.8, -2.5, 2.0, -3.8])}
base = 70 + np.array([school_effect[s] for s in school_id]) + rng.normal(0, 9, n)

math    = np.clip(base + rng.normal(0, 6, n) - 1.5, 25, 100)
reading = np.clip(base + rng.normal(0, 5.5, n) + 1.0, 28, 100)
science = np.clip(base + rng.normal(0, 7, n) - 3.0, 20, 100)

frpl = rng.choice([False, True], n, p=[0.52, 0.48])
ell  = rng.choice([False, True], n, p=[0.81, 0.19])
sped = rng.choice([False, True], n, p=[0.87, 0.13])

math    = np.clip(math    - 4.5*frpl - 3.0*ell - 5.5*sped, 20, 100)
reading = np.clip(reading - 3.8*frpl - 5.5*ell - 4.0*sped, 20, 100)
science = np.clip(science - 4.2*frpl - 2.5*ell - 6.0*sped, 20, 100)

students = pd.DataFrame({
    "school": school_id, "frpl": frpl, "ell": ell, "sped": sped,
    "math": math.round(1), "reading": reading.round(1), "science": science.round(1)
})
students["avg_score"] = students[["math", "reading", "science"]].mean(axis=1).round(1)
students["at_risk"] = students["avg_score"] < 55

scores_long = students.melt(
    id_vars=["school", "frpl", "ell", "sped", "at_risk"],
    value_vars=["math", "reading", "science"],
    var_name="subject", value_name="score"
)

# ----- Visualization -----
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

sns.histplot(data=scores_long, x="score", hue="subject", kde=True,
             element="step", stat="density", common_norm=False, ax=axes[0])
axes[0].set_title("Score Distributions by Subject")
axes[0].set_xlabel("Score")

sns.violinplot(data=scores_long, x="subject", y="score", inner="quartile",
               palette="Set2", ax=axes[1])
axes[1].set_title("Score Distribution Shape by Subject")
axes[1].set_xlabel("")

plt.tight_layout()
plt.show()


### Full Block 2 — FRPL Achievement Gap (Point Plot + Violin)


In [ ]:
# ================================================================
# FULL CODE: FRPL Achievement Gap Visualizations
# Requires: scores_long DataFrame from Block 1 (or earlier cells)
# ================================================================
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Split violin
sns.violinplot(data=scores_long, x="subject", y="score", hue="frpl",
               split=True, inner="quart",
               palette={False: "#4c72b0", True: "#c44e52"}, ax=axes[0])
axes[0].set_title("Subject Scores by FRPL Status (split violin)")
axes[0].set_xlabel("")
axes[0].legend(title="FRPL", labels=["No", "Yes"])

# Point plot with confidence intervals
sns.pointplot(data=scores_long, x="subject", y="score", hue="frpl",
              errorbar=("ci", 95), dodge=True, markers=["o", "s"],
              palette={False: "#4c72b0", True: "#c44e52"}, ax=axes[1])
axes[1].set_title("Mean Score ± 95% CI by Subject and FRPL")
axes[1].set_xlabel("")
axes[1].legend(title="FRPL")

plt.tight_layout()
plt.show()


### Full Block 3 — Complete Heatmap Suite (4 education heatmaps)


In [ ]:
# ================================================================
# FULL CODE: Four practical education heatmaps
# Requires: students and scores_long DataFrames
# ================================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Consistent school order (best → worst)
school_order = students.groupby("school")["avg_score"].mean().sort_values(ascending=False).index

fig, axes = plt.subplots(2, 2, figsize=(14, 11))

# ----- 1. Mean Score Heatmap (School × Subject) -----
school_subj = (scores_long
               .pivot_table(values="score", index="school", columns="subject", aggfunc="mean")
               .reindex(school_order))
sns.heatmap(school_subj, annot=True, fmt=".1f", cmap="YlGnBu",
            linewidths=0.5, ax=axes[0, 0], cbar_kws={"label": "Mean Score"})
axes[0, 0].set_title("Mean Score by School × Subject")
axes[0, 0].set_xlabel("")
axes[0, 0].set_ylabel("")

# ----- 2. Proficiency Rate Heatmap -----
prof = scores_long.copy()
prof["proficient"] = prof["score"] >= 70
prof_rate = (prof.groupby(["school", "subject"])["proficient"]
             .mean().unstack().reindex(school_order) * 100)
sns.heatmap(prof_rate, annot=True, fmt=".0f", cmap="YlGnBu",
            linewidths=0.5, ax=axes[0, 1], cbar_kws={"label": "% Proficient"})
axes[0, 1].set_title("Proficiency Rate (%) — cut-score 70")
axes[0, 1].set_xlabel("")
axes[0, 1].set_ylabel("")

# ----- 3. FRPL Gap Heatmap -----
gap_data = (scores_long.groupby(["school", "subject", "frpl"])["score"]
            .mean().unstack("frpl"))
# Handle both boolean and string column names
col_non = False if False in gap_data.columns else "False"
col_frpl = True if True in gap_data.columns else "True"
gap_data["gap"] = gap_data[col_non] - gap_data[col_frpl]
gap_matrix = gap_data["gap"].unstack("subject").reindex(school_order)
sns.heatmap(gap_matrix, annot=True, fmt=".1f", cmap="RdYlGn_r", center=0,
            linewidths=0.5, ax=axes[1, 0], cbar_kws={"label": "Gap (points)"})
axes[1, 0].set_title("FRPL Achievement Gap (Non-FRPL - FRPL)")
axes[1, 0].set_xlabel("")
axes[1, 0].set_ylabel("")

# ----- 4. At-Risk Rate by School × Group -----
students = students.copy()
students["group"] = np.select(
    [students["frpl"] & students["ell"],
     students["frpl"] & ~students["ell"],
     ~students["frpl"] & students["ell"],
     ~students["frpl"] & ~students["ell"]],
    ["FRPL + ELL", "FRPL only", "ELL only", "Neither"],
    default="Other"
)
risk = (students.groupby(["school", "group"])["at_risk"]
        .mean().unstack().reindex(school_order) * 100)
col_order = [c for c in ["Neither", "ELL only", "FRPL only", "FRPL + ELL"] if c in risk.columns]
risk = risk[col_order]
sns.heatmap(risk, annot=True, fmt=".0f", cmap="OrRd",
            linewidths=0.5, ax=axes[1, 1], cbar_kws={"label": "% At-Risk"})
axes[1, 1].set_title("At-Risk Rate (%) by School × Student Group")
axes[1, 1].set_xlabel("")
axes[1, 1].set_ylabel("")

plt.tight_layout()
plt.show()


### Full Block 4 — Correlation Heatmap + Regression


In [ ]:
# ================================================================
# FULL CODE: Correlation heatmap (lower triangle) + Math vs Reading
# Requires: students DataFrame
# ================================================================
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

corr = students[["math", "reading", "science", "avg_score"]].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="RdBu_r", center=0,
            square=True, linewidths=1.2, linecolor="white",
            cbar_kws={"shrink": 0.8, "label": "Pearson r"},
            annot_kws={"size": 11}, ax=axes[0])
axes[0].set_title("Subject Correlations (lower triangle)")

sns.regplot(data=students, x="math", y="reading",
            scatter_kws={"alpha": 0.35, "s": 18},
            line_kws={"color": "darkred"}, ax=axes[1])
axes[1].set_title("Math vs Reading with Linear Fit")
axes[1].set_xlabel("Math Score")
axes[1].set_ylabel("Reading Score")

plt.tight_layout()
plt.show()


### Full Block 5 — Compact Equity & Performance Dashboard


In [ ]:
# ================================================================
# FULL CODE: Four-panel education dashboard
# Requires: students and scores_long DataFrames
# ================================================================
import matplotlib.pyplot as plt
import seaborn as sns

school_order = students.groupby("school")["avg_score"].mean().sort_values(ascending=False).index

fig, axes = plt.subplots(2, 2, figsize=(13, 10))

# 1. Density by subject
sns.kdeplot(data=scores_long, x="score", hue="subject", fill=True,
            common_norm=False, alpha=0.4, ax=axes[0, 0])
axes[0, 0].set_title("Score Density by Subject")
axes[0, 0].set_xlabel("Score")

# 2. FRPL gap
sns.pointplot(data=scores_long, x="subject", y="score", hue="frpl",
              errorbar=("ci", 95), dodge=True, ax=axes[0, 1],
              palette={False: "#4c72b0", True: "#c44e52"})
axes[0, 1].set_title("FRPL Achievement Gap by Subject")
axes[0, 1].set_xlabel("")
axes[0, 1].legend(title="FRPL")

# 3. School ranking
sns.barplot(data=students, x="avg_score", y="school", order=school_order,
            palette="Blues_r", errorbar=("ci", 95), ax=axes[1, 0])
axes[1, 0].set_title("Schools Ranked by Mean Average Score")
axes[1, 0].set_xlabel("Mean Score")
axes[1, 0].set_ylabel("")

# 4. At-risk heatmap
risk_pivot = (students.pivot_table(values="at_risk", index="school",
                                   columns="frpl", aggfunc="mean")
              .reindex(school_order))
risk_pivot.columns = ["Non-FRPL", "FRPL"]
sns.heatmap(risk_pivot, annot=True, fmt=".1%", cmap="OrRd", ax=axes[1, 1])
axes[1, 1].set_title("At-Risk Rate by School × FRPL")
axes[1, 1].set_xlabel("")

plt.tight_layout()
plt.show()
